In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailpulse.silver;

In [0]:
%sql
SHOW SCHEMAS IN retailpulse;

In [0]:
%sql
DESCRIBE retailpulse.bronze.customers;

In [0]:

from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

source_table = "retailpulse.bronze.customers"
target_table = "retailpulse.silver.customers"
rejected_table = "retailpulse.silver.customers_rejected"

primary_key_column = "customer_id"


# ============================================================
# 2. DEFINE CUSTOMER SCHEMA
# ============================================================

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), False),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), False),
    StructField("source_file", StringType(), True),
    StructField("ingested_at", TimestampType(), True),
    StructField("run_id", StringType(), True)
])


# ============================================================
# 3. READ BRONZE DATA
# ============================================================

customers_df = spark.read.table(source_table)

customers_df.limit(10).display()
customers_df.printSchema()

total_records_count = customers_df.count()

print("Total Bronze records:", total_records_count)


# ============================================================
# 4. STRING STANDARDIZATION
# ============================================================

string_columns = [
    field.name
    for field in customers_df.schema.fields
    if isinstance(field.dataType, StringType)
]

# Remove leading and trailing spaces
for column_name in string_columns:
    customers_df = customers_df.withColumn(
        column_name,
        F.trim(F.col(column_name))
    )

# Standardize city and state
customers_df = customers_df.withColumns({
    "customer_city": F.initcap(F.col("customer_city")),
    "customer_state": F.upper(F.col("customer_state"))
})

customers_df.limit(10).display()


# ============================================================
# 5. DATA QUALITY PROFILING
# ============================================================

# NULL count for every column
null_count_expressions = [
    F.coalesce(
        F.sum(
            F.when(F.col(column_name).isNull(), 1).otherwise(0)
        ),
        F.lit(0)
    ).alias(column_name)
    for column_name in customers_df.columns
]

null_counts_dict = (
    customers_df.agg(*null_count_expressions).first().asDict()
)


# Empty-string count for string columns
empty_string_expressions = [
    F.coalesce(
        F.sum(
            F.when(F.col(column_name) == "", 1).otherwise(0)
        ),
        F.lit(0)
    ).alias(column_name)
    for column_name in string_columns
]

empty_string_counts_dict = (
    customers_df.agg(*empty_string_expressions).first().asDict()
)

print("NULL COUNTS:", null_counts_dict)
print("EMPTY STRING COUNTS:", empty_string_counts_dict)


# ============================================================
# 6. MANDATORY FIELD VALIDATION
# ============================================================

mandatory_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_state"
]

invalid_records_df = customers_df.filter(
    (F.col("customer_id").isNull()) |
    (F.col("customer_id") == "") |
    (F.col("customer_unique_id").isNull()) |
    (F.col("customer_unique_id") == "") |
    (F.col("customer_state").isNull()) |
    (F.col("customer_state") == "")
)

valid_records_df = customers_df.filter(
    (F.col("customer_id").isNotNull()) &
    (F.col("customer_id") != "") &
    (F.col("customer_unique_id").isNotNull()) &
    (F.col("customer_unique_id") != "") &
    (F.col("customer_state").isNotNull()) &
    (F.col("customer_state") != "")
)


# ============================================================
# 7. RECORD COUNTS AND DATA RECONCILIATION
# ============================================================

invalid_records_count = invalid_records_df.count()
valid_records_count = valid_records_df.count()

reconciled_count = valid_records_count + invalid_records_count

print("Bronze records:", total_records_count)
print("Valid records:", valid_records_count)
print("Invalid records:", invalid_records_count)
print("Reconciled records:", reconciled_count)

if total_records_count == reconciled_count:
    print("RECONCILIATION PASSED")
else:
    print("RECONCILIATION FAILED")


# ============================================================
# 8. PRIMARY KEY DUPLICATE ANALYSIS
# ============================================================

duplicate_customer_ids_df = (
    valid_records_df
    .groupBy(primary_key_column)
    .count()
    .filter(F.col("count") > 1)
)

# Number of duplicate customer ID groups
duplicate_customer_id_group_count = duplicate_customer_ids_df.count()

# Total records belonging to duplicate groups
duplicate_customer_record_count = (
    duplicate_customer_ids_df
    .agg(F.sum("count").alias("duplicate_record_count"))
    .first()["duplicate_record_count"] or 0
)

# Maximum occurrences of a single customer ID across all valid IDs.
# This returns 1 when there are no duplicate IDs (not 0).
customer_id_frequency_df = (
    valid_records_df
    .groupBy(primary_key_column)
    .count()
)

max_customer_id_occurrence = (
    customer_id_frequency_df
    .agg(F.coalesce(F.max("count"), F.lit(0)).alias("max_occurrence"))
    .first()["max_occurrence"]
)

print("Duplicate customer ID groups:", duplicate_customer_id_group_count)
print("Records in duplicate groups:", duplicate_customer_record_count)
print("Maximum customer ID occurrence:", max_customer_id_occurrence)


# ============================================================
# 9. BUSINESS-LEVEL EXACT DUPLICATE ANALYSIS
# ============================================================

business_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

exact_duplicates_df = (
    valid_records_df
    .groupBy(*business_columns)
    .count()
    .filter(F.col("count") > 1)
)

business_duplicate_metrics = exact_duplicates_df.agg(
    F.count("*").alias("duplicate_group_count"),
    F.coalesce(
        F.sum("count"), F.lit(0)
    ).alias("duplicate_record_count"),
    F.coalesce(
        F.sum(F.col("count") - 1), F.lit(0)
    ).alias("duplicate_excess_count")
).first()

exact_duplicate_business_group_count = (
    business_duplicate_metrics["duplicate_group_count"]
)

exact_duplicate_business_record_count = (
    business_duplicate_metrics["duplicate_record_count"]
)

exact_duplicate_business_excess_count = (
    business_duplicate_metrics["duplicate_excess_count"]
)


# ============================================================
# 10. FULL-RECORD EXACT DUPLICATE ANALYSIS
# ============================================================

exact_duplicate_full_records_df = (
    valid_records_df
    .groupBy(*valid_records_df.columns)
    .count()
    .filter(F.col("count") > 1)
)

full_records_duplicate_metrics = exact_duplicate_full_records_df.agg(
    F.count("*").alias("duplicate_group_count"),
    F.coalesce(
        F.sum("count"), F.lit(0)
    ).alias("duplicate_record_count"),
    F.coalesce(
        F.sum(F.col("count") - 1), F.lit(0)
    ).alias("duplicate_excess_count")
).first()

exact_duplicate_full_group_count = (
    full_records_duplicate_metrics["duplicate_group_count"]
)

exact_duplicate_full_record_count = (
    full_records_duplicate_metrics["duplicate_record_count"]
)

exact_duplicate_full_excess_count = (
    full_records_duplicate_metrics["duplicate_excess_count"]
)


# ============================================================
# 11. DUPLICATE ANALYSIS SUMMARY
# ============================================================

print("\n========== PRIMARY KEY DUPLICATES ==========")
print("Duplicate ID groups:", duplicate_customer_id_group_count)
print("Duplicate records:", duplicate_customer_record_count)
print("Maximum ID occurrence:", max_customer_id_occurrence)

print("\n========== BUSINESS-LEVEL DUPLICATES ==========")
print("Duplicate groups:", exact_duplicate_business_group_count)
print("Duplicate records:", exact_duplicate_business_record_count)
print("Excess duplicates:", exact_duplicate_business_excess_count)

print("\n========== FULL-RECORD DUPLICATES ==========")
print("Duplicate groups:", exact_duplicate_full_group_count)
print("Duplicate records:", exact_duplicate_full_record_count)
print("Excess duplicates:", exact_duplicate_full_excess_count)


# ============================================================
# 12. CUSTOMER STATE VALIDATION
# ============================================================

# Step 1: Get distinct customer states

distinct_state_df = valid_records_df.select(
    "customer_state"
).distinct()


# Step 2: Count customers belonging to each state

each_customer_state_count = valid_records_df.groupBy(
    "customer_state"
).count()

print("\n========== CUSTOMER STATE DISTRIBUTION ==========")
display(each_customer_state_count)


# Step 3: Define expected valid Brazilian states

states = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES",
    "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR",
    "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC",
    "SP", "SE", "TO"
]


# Step 4: Convert expected states into a Spark DataFrame

state_data = []

for state in states:
    state_data.append((state,))

expected_states_df = spark.createDataFrame(
    state_data,
    ["customer_state"]
)


# Step 5: Identify invalid states using Left Anti Join

invalid_state_df = distinct_state_df.join(
    expected_states_df,
    on="customer_state",
    how="left_anti"
)


# Step 6: Count distinct invalid states

invalid_states_count = invalid_state_df.count()


# Step 7: Display invalid states

print("\n========== INVALID CUSTOMER STATES ==========")
display(invalid_state_df)


# Step 8: Identify customer records having invalid states

invalid_customer_state_df = valid_records_df.join(
    invalid_state_df,
    on="customer_state",
    how="inner"
)


# Step 9: Count invalid customer records

invalid_customer_state_record_count = (
    invalid_customer_state_df.count()
)


# Step 10: Calculate valid customer records

total_customer_records = valid_records_df.count()

valid_customer_state_record_count = (
    total_customer_records - invalid_customer_state_record_count
)


# Step 11: Customer state validation summary

print("\n========== CUSTOMER STATE VALIDATION SUMMARY ==========")

print("Expected distinct states:", len(states))
print("Actual distinct states:", distinct_state_df.count())
print("Invalid distinct states:", invalid_states_count)

print("Total customer records:", total_customer_records)
print("Valid customer state records:", valid_customer_state_record_count)
print("Invalid customer state records:", invalid_customer_state_record_count)


# ============================================================
# 13. CUSTOMER ZIP CODE PREFIX PROFILING
# ============================================================

# Step 1: Inspect ZIP prefix data type

customer_zip_code_prefix_datatype = (
    valid_records_df.schema["customer_zip_code_prefix"]
)

print("\n========== CUSTOMER ZIP CODE PREFIX DATA TYPE ==========")
print(customer_zip_code_prefix_datatype)


# Step 2: Identify NULL ZIP prefixes

customer_zip_code_prefix_nulls_df = valid_records_df.filter(
    F.col("customer_zip_code_prefix").isNull()
)

customer_zip_code_prefix_nullcount = (
    customer_zip_code_prefix_nulls_df.count()
)

print("\n========== CUSTOMER ZIP CODE PREFIX NULL COUNT ==========")
print(customer_zip_code_prefix_nullcount)


# Step 3: Identify blank ZIP prefixes

customer_zip_code_prefix_blanks_df = valid_records_df.filter(
    F.trim(F.col("customer_zip_code_prefix")) == ""
)

customer_zip_code_prefix_blank_count = (
    customer_zip_code_prefix_blanks_df.count()
)

print("\n========== CUSTOMER ZIP CODE PREFIX BLANK COUNT ==========")
print(customer_zip_code_prefix_blank_count)


# Step 4: Identify distinct ZIP prefixes

distinct_customer_zip_code_prefix_df = (
    valid_records_df
    .select("customer_zip_code_prefix")
    .distinct()
)

distinct_customer_zip_code_prefix_count = (
    distinct_customer_zip_code_prefix_df.count()
)

print("\n========== DISTINCT CUSTOMER ZIP CODE PREFIX COUNT ==========")
print(distinct_customer_zip_code_prefix_count)


# ============================================================
# 14. CUSTOMER ZIP CODE PREFIX LENGTH VALIDATION
# ============================================================

# Step 1: Calculate ZIP prefix length distribution

zip_prefix_length_distribution_df = (
    valid_records_df
    .withColumn(
        "zip_prefix_length",
        F.length(F.col("customer_zip_code_prefix"))
    )
    .groupBy("zip_prefix_length")
    .count()
)

print("\n========== CUSTOMER ZIP CODE PREFIX LENGTH DISTRIBUTION ==========")
display(zip_prefix_length_distribution_df)


# Step 2: Identify ZIP prefixes whose length is not 5

invalid_zip_length_df = (
    valid_records_df
    .withColumn(
        "zip_prefix_length",
        F.length(F.col("customer_zip_code_prefix"))
    )
    .filter(F.col("zip_prefix_length") != 5)
)

invalid_zip_length_record_count = invalid_zip_length_df.count()

print("\n========== INVALID ZIP CODE PREFIX LENGTH COUNT ==========")
print(invalid_zip_length_record_count)


# Step 3: Display invalid ZIP prefix records

print("\n========== INVALID ZIP CODE PREFIX RECORDS ==========")
display(invalid_zip_length_df)


# Step 4: Identify distinct invalid ZIP prefix values

invalid_zip_length_values_df = (
    invalid_zip_length_df
    .select("customer_zip_code_prefix")
    .distinct()
)

print("\n========== DISTINCT INVALID ZIP PREFIX VALUES ==========")
display(invalid_zip_length_values_df)


# ============================================================
# 15. CUSTOMER ZIP CODE NUMERIC-FORMAT VALIDATION
# ============================================================

# Step 1: Identify ZIP prefixes containing non-numeric characters

invalid_zip_format_df = valid_records_df.filter(
    (F.col("customer_zip_code_prefix").isNotNull()) &
    (F.trim(F.col("customer_zip_code_prefix")) != "") &
    (~F.col("customer_zip_code_prefix").rlike("^[0-9]+$"))
)


# Step 2: Count invalid ZIP format records

invalid_zip_format_record_count = (
    invalid_zip_format_df.count()
)

print("\n========== INVALID ZIP FORMAT RECORD COUNT ==========")
print(invalid_zip_format_record_count)


# Step 3: Identify distinct invalid ZIP prefix values

invalid_zip_prefix_values_df = (
    invalid_zip_format_df
    .select("customer_zip_code_prefix")
    .distinct()
)

invalid_zip_prefix_value_count = (
    invalid_zip_prefix_values_df.count()
)

print("\n========== DISTINCT INVALID ZIP PREFIX VALUES ==========")
print(invalid_zip_prefix_value_count)

display(invalid_zip_prefix_values_df)


# Step 4: Display invalid customer records

print("\n========== INVALID ZIP FORMAT CUSTOMER RECORDS ==========")
display(invalid_zip_format_df)


# Step 5: ZIP numeric-format validation summary

print("\n========== ZIP NUMERIC-FORMAT VALIDATION SUMMARY ==========")

print("Total customer records:", valid_records_df.count())

print(
    "Invalid ZIP format records:",
    invalid_zip_format_record_count
)

print(
    "Distinct invalid ZIP prefixes:",
    invalid_zip_prefix_value_count
)


# ============================================================
# 16. CONSOLIDATED CUSTOMER ZIP CODE PREFIX VALIDATION
# ============================================================

# ZIP prefix is optional in the current customer schema.
# NULL and blank values are therefore profiled separately and are NOT
# classified as invalid format records in this section.
# For populated ZIP prefixes, require exactly 5 digits.

zip_prefix_populated_condition = (
    F.col("customer_zip_code_prefix").isNotNull()
    & (F.trim(F.col("customer_zip_code_prefix")) != "")
)

zip_prefix_invalid_condition = (
    zip_prefix_populated_condition
    & (
        (F.length(F.col("customer_zip_code_prefix")) != 5)
        | (~F.col("customer_zip_code_prefix").rlike("^[0-9]{5}$"))
    )
)

invalid_zip_records_df = valid_records_df.filter(
    zip_prefix_invalid_condition
)

valid_zip_records_df = valid_records_df.filter(
    ~zip_prefix_invalid_condition
)

invalid_zip_record_count = invalid_zip_records_df.count()
valid_zip_record_count = valid_zip_records_df.count()

# Distinct invalid values help identify the source-data issue.
invalid_zip_values_df = (
    invalid_zip_records_df
    .select("customer_zip_code_prefix")
    .distinct()
)

invalid_zip_value_count = invalid_zip_values_df.count()

# Missing ZIP prefixes are reported separately because the field is optional.
missing_zip_record_count = valid_records_df.filter(
    F.col("customer_zip_code_prefix").isNull()
    | (F.trim(F.col("customer_zip_code_prefix")) == "")
).count()

zip_reconciled_count = valid_zip_record_count + invalid_zip_record_count

print("\n========== CONSOLIDATED ZIP VALIDATION SUMMARY ==========")
print("Total customer records:", valid_records_df.count())
print("Missing ZIP prefix records (NULL or blank):", missing_zip_record_count)
print("Invalid populated ZIP prefix records:", invalid_zip_record_count)
print("Distinct invalid ZIP prefix values:", invalid_zip_value_count)
print("Valid ZIP prefix records (including missing/blank optional values):", valid_zip_record_count)
print("Reconciled customer records:", zip_reconciled_count)

if valid_records_df.count() == zip_reconciled_count:
    print("ZIP VALIDATION RECONCILIATION PASSED")
else:
    print("ZIP VALIDATION RECONCILIATION FAILED")

print("\n========== INVALID ZIP PREFIX RECORDS ==========")
display(invalid_zip_records_df)

print("\n========== DISTINCT INVALID ZIP PREFIX VALUES ==========")
display(invalid_zip_values_df)





In [0]:

# ============================================================
# 1. CUSTOMER CITY DATA TYPE
# ============================================================

customer_city_schema = valid_records_df.schema["customer_city"]

print("\n========== CUSTOMER CITY DATA TYPE ==========")
print(customer_city_schema.dataType)


# ============================================================
# 2. CUSTOMER CITY NULL VALIDATION
# ============================================================

customer_city_nulls_df = valid_records_df.filter(
    F.col("customer_city").isNull()
)

customer_city_nulls_count = customer_city_nulls_df.count()

print("\n========== CUSTOMER CITY NULL VALIDATION ==========")
print("NULL city records:", customer_city_nulls_count)

display(customer_city_nulls_df)


# ============================================================
# 3. CUSTOMER CITY BLANK VALIDATION
# ============================================================

customer_city_blanks_df = valid_records_df.filter(
    F.trim(F.col("customer_city")) == ""
)

customer_city_blanks_count = customer_city_blanks_df.count()

print("\n========== CUSTOMER CITY BLANK VALIDATION ==========")
print("Blank city records:", customer_city_blanks_count)

display(customer_city_blanks_df)


# ============================================================
# 4. DISTINCT CUSTOMER CITY COUNT
# ============================================================

customer_city_distinct_df = (
    valid_records_df
    .select("customer_city")
    .distinct()
)

customer_city_distinct_count = customer_city_distinct_df.count()

print("\n========== DISTINCT CUSTOMER CITIES ==========")
print("Distinct city count:", customer_city_distinct_count)

display(customer_city_distinct_df)


# ============================================================
# 5. TOP 20 CUSTOMER CITIES
# ============================================================

top_20_customer_cities = (
    valid_records_df
    .groupBy("customer_city")
    .count()
    .orderBy(F.col("count").desc())
    .limit(20)
)

print("\n========== TOP 20 CUSTOMER CITIES ==========")
display(top_20_customer_cities)


# ============================================================
# 6. BOTTOM 20 CUSTOMER CITIES
# ============================================================

below_20_customer_cities = (
    valid_records_df
    .groupBy("customer_city")
    .count()
    .orderBy(F.col("count").asc_nulls_last())
    .limit(20)
)

print("\n========== BOTTOM 20 CUSTOMER CITIES ==========")
display(below_20_customer_cities)


# ============================================================
# 7. CUSTOMER CITY NUMERIC-CHARACTER VALIDATION
# ============================================================

city_names_contains_digits_df = valid_records_df.filter(
    F.col("customer_city").rlike("[0-9]")
)

city_names_contains_digits_record_count = (
    city_names_contains_digits_df.count()
)

city_names_contains_digits_distinct_df = (
    city_names_contains_digits_df
    .select("customer_city")
    .distinct()
)

city_names_contains_digits_distinct_count = (
    city_names_contains_digits_distinct_df.count()
)

print("\n========== CITY NUMERIC-CHARACTER VALIDATION ==========")
print(
    "Records containing digits:",
    city_names_contains_digits_record_count
)
print(
    "Distinct city names containing digits:",
    city_names_contains_digits_distinct_count
)

display(city_names_contains_digits_distinct_df)


# ============================================================
# 8. CUSTOMER CITY SPECIAL-CHARACTER VALIDATION
# ============================================================

# Allow Unicode letters, spaces, hyphens and apostrophes.
# This supports accented Brazilian city names.

city_names_contains_special_chars_df = valid_records_df.filter(
    ~F.col("customer_city").rlike(r"^[\p{L} '-]+$")
)

city_names_contains_special_chars_record_count = (
    city_names_contains_special_chars_df.count()
)

city_names_contains_special_chars_distinct_df = (
    city_names_contains_special_chars_df
    .select("customer_city")
    .distinct()
)

city_names_contains_special_chars_distinct_count = (
    city_names_contains_special_chars_distinct_df.count()
)

print("\n========== CITY SPECIAL-CHARACTER VALIDATION ==========")
print(
    "Records containing unexpected characters:",
    city_names_contains_special_chars_record_count
)
print(
    "Distinct city names containing unexpected characters:",
    city_names_contains_special_chars_distinct_count
)

display(city_names_contains_special_chars_distinct_df)


# ============================================================
# 9. CUSTOMER CITY VALIDATION SUMMARY
# ============================================================

print("\n========== CUSTOMER CITY PROFILING SUMMARY ==========")

print("Total customer records:", valid_records_df.count())
print("NULL city records:", customer_city_nulls_count)
print("Blank city records:", customer_city_blanks_count)
print("Distinct city names:", customer_city_distinct_count)

print(
    "Records containing digits:",
    city_names_contains_digits_record_count
)
print(
    "Distinct city names containing digits:",
    city_names_contains_digits_distinct_count
)

print(
    "Records containing unexpected characters:",
    city_names_contains_special_chars_record_count
)
print(
    "Distinct city names containing unexpected characters:",
    city_names_contains_special_chars_distinct_count
)

# ============================================================
# 1. CUSTOMER CITY CAPITALIZATION VALIDATION
# ============================================================

customer_city_capitalization_issues_df = (
    valid_records_df
    .withColumn(
        "standardized_city",
        F.initcap(F.col("customer_city"))
    )
    .filter(
        F.col("customer_city") != F.col("standardized_city")
    )
)

customer_city_capitalization_issue_count = (
    customer_city_capitalization_issues_df.count()
)

print(
    "Records with capitalization differences:",
    customer_city_capitalization_issue_count
)

customer_city_capitalization_distinct_df = (
    customer_city_capitalization_issues_df
    .select("customer_city")
    .distinct()
)

customer_city_capitalization_distinct_count = (
    customer_city_capitalization_distinct_df.count()
)

print(
    "Distinct city names with capitalization differences:",
    customer_city_capitalization_distinct_count
)

customer_city_capitalization_comparison_df = (
    customer_city_capitalization_issues_df
    .select(
        "customer_city",
        "standardized_city"
    )
    .distinct()
    .orderBy("customer_city")
)

display(customer_city_capitalization_comparison_df)



# ============================================================
# 1. EXTRACT DISTINCT CITY-STATE COMBINATIONS
# ============================================================

distinct_city_state_df = (
    valid_records_df
    .select(
        "customer_city",
        "customer_state"
    )
    .distinct()
)

print("\n========== DISTINCT CITY-STATE COMBINATIONS ==========")

distinct_city_state_count = distinct_city_state_df.count()

print("Distinct city-state combinations:", distinct_city_state_count)

display(distinct_city_state_df)


# ============================================================
# 2. COUNT DISTINCT STATES ASSOCIATED WITH EACH CITY
# ============================================================

city_state_count_df = (
    distinct_city_state_df
    .groupBy("customer_city")
    .agg(
        F.countDistinct("customer_state").alias("state_count")
    )
)

print("\n========== STATES ASSOCIATED WITH EACH CITY ==========")

display(city_state_count_df)


# ============================================================
# 3. IDENTIFY CITIES ASSOCIATED WITH MULTIPLE STATES
# ============================================================

multiple_state_city_df = (
    city_state_count_df
    .filter(F.col("state_count") > 1)
)

multiple_state_city_count = multiple_state_city_df.count()

print("\n========== CITIES ASSOCIATED WITH MULTIPLE STATES ==========")

print("Multiple-state city count:", multiple_state_city_count)

display(multiple_state_city_df)


# ============================================================
# 4. DISPLAY ACTUAL CITY-STATE COMBINATIONS
# ============================================================

multiple_state_cities_df = (
    distinct_city_state_df
    .join(
        multiple_state_city_df,
        on="customer_city",
        how="inner"
    )
    .select(
        "customer_city",
        "customer_state",
        "state_count"
    )
    .orderBy(
        "customer_city",
        "customer_state"
    )
)

print("\n========== MULTIPLE-STATE CITY COMBINATIONS ==========")

display(multiple_state_cities_df)


# ============================================================
# 5. COUNT AFFECTED CUSTOMER RECORDS
# ============================================================

affected_customer_records_df = (
    valid_records_df
    .join(
        multiple_state_city_df.select("customer_city"),
        on="customer_city",
        how="inner"
    )
)

affected_customer_record_count = affected_customer_records_df.count()

affected_customer_distinct_city_count = (
    affected_customer_records_df
    .select("customer_city")
    .distinct()
    .count()
)

print("\n========== AFFECTED CUSTOMER RECORDS ==========")

print("Affected customer records:", affected_customer_record_count)

print(
    "Distinct affected city names:",
    affected_customer_distinct_city_count
)

display(
    affected_customer_records_df.select(
        "customer_id",
        "customer_city",
        "customer_state"
    )
)


# ============================================================
# 6. CITY-STATE VALIDATION SUMMARY
# ============================================================

print("\n========== CITY-STATE VALIDATION SUMMARY ==========")

print("Total customer records:", valid_records_df.count())

print(
    "Distinct city-state combinations:",
    distinct_city_state_count
)

print(
    "Cities associated with multiple states:",
    multiple_state_city_count
)

print(
    "Affected customer records:",
    affected_customer_record_count
)

print(
    "Distinct affected city names:",
    affected_customer_distinct_city_count
)
